# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My rule and its reason codes

I will use a simple weighted baseline score to rank pages for review.

The score combines four observable signals:

- 40% visibility: pages with more observed impressions have more evidence and potential value.
- 30% freshness risk: older pages that have not been updated recently receive more review weight.
- 25% position opportunity: pages with an observed search position in the useful range receive more weight.
- 5% depth gap: thinner pages receive a small additional review weight.

The score is a prioritization rule, not a prediction of whether a refresh will definitely succeed.

Reason codes explain why a page was selected:

- `stale_visible_page`
- `declining_with_demand`
- `thin_visible_page`
- `page_one_decay_risk`
- `low_ctr_visible_page`
- `low_engagement_visible_page`

A page can receive more than one reason code. The final queue is ordered by the baseline score from highest to lowest.

In [1]:
!git clone https://github.com/Alpeshmore/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 143 (delta 53), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.86 MiB | 4.25 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [2]:
import pandas as pd
import numpy as np

DATA_PATH = "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
# ML-07 — Section 1
# Check that the required columns exist before building the rule.

required_columns = [
    "content_id",
    "impressions_90d",
    "sessions_90d",
    "word_count",
    "days_since_last_update",
    "content_age_days",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "trend_direction",
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("Missing required columns:", missing_columns)

assert not missing_columns, "Required columns are missing."

reason_codes = [
    "stale_visible_page",
    "declining_with_demand",
    "thin_visible_page",
    "page_one_decay_risk",
    "low_ctr_visible_page",
    "low_engagement_visible_page",
]

print("Reason codes:")
for code in reason_codes:
    print("-", code)


Missing required columns: []
Reason codes:
- stale_visible_page
- declining_with_demand
- thin_visible_page
- page_one_decay_risk
- low_ctr_visible_page
- low_engagement_visible_page


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Build the ranked queue

I calculate the baseline score from observable current-window measurements.

The score is used only to prioritize human review. It does not claim that a refresh will cause improvement.

I normalize the four components to comparable 0–1 ranges before applying the weights. The output contains the content identifier, score, action, and reason codes needed for review.

In [4]:
# ML-07 — Section 2
# Build the baseline action score and ranked queue.

import numpy as np
import pandas as pd
from pathlib import Path

score_df = df.copy()

# -----------------------------
# 1. Visibility score
# -----------------------------
# More impressions = more observed visibility/evidence.
# log1p reduces the effect of very large values.

score_df["visibility_score"] = (
    np.log1p(score_df["impressions_90d"].clip(lower=0))
)

def minmax(series):
    series = series.astype(float)
    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(0.0, index=series.index)

    return (series - minimum) / (maximum - minimum)


score_df["visibility_score"] = minmax(
    score_df["visibility_score"]
)

# -----------------------------
# 2. Freshness risk
# -----------------------------
score_df["freshness_risk_score"] = minmax(
    score_df["days_since_last_update"].clip(lower=0)
)

# -----------------------------
# 3. Position opportunity
# -----------------------------
# Pages with observed positions 1–20 receive stronger opportunity weight.
# Outside that range, the score decreases.

position = pd.to_numeric(
    score_df["avg_position"],
    errors="coerce"
)

score_df["position_opportunity_score"] = np.where(
    position.between(1, 20),
    (21 - position) / 20,
    0
)

score_df["position_opportunity_score"] = (
    score_df["position_opportunity_score"]
    .fillna(0)
    .clip(0, 1)
)

# -----------------------------
# 4. Depth gap
# -----------------------------
# Smaller pages receive more depth-gap weight.
# 1200 words is used as the simple starter threshold.

word_count = pd.to_numeric(
    score_df["word_count"],
    errors="coerce"
).fillna(0)

score_df["depth_gap_score"] = (
    (1200 - word_count) / 1200
).clip(0, 1)

# -----------------------------
# 5. Weighted baseline score
# -----------------------------

score_df["baseline_action_score"] = (
    0.40 * score_df["visibility_score"]
    + 0.30 * score_df["freshness_risk_score"]
    + 0.25 * score_df["position_opportunity_score"]
    + 0.05 * score_df["depth_gap_score"]
)

# Convert to 0–100 for easier reading.
score_df["baseline_action_score"] = (
    100 * score_df["baseline_action_score"]
)

# -----------------------------
# 6. Reason codes
# -----------------------------

def get_reason_codes(row):
    reasons = []

    if (
        row["days_since_last_update"] >= 180
        and row["impressions_90d"] >= 500
    ):
        reasons.append("stale_visible_page")

    if (
        str(row["trend_direction"]).lower() == "down"
        and row["impressions_90d"] >= 100
    ):
        reasons.append("declining_with_demand")

    if (
        row["word_count"] > 0
        and row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    if (
        row["avg_position"] > 0
        and row["avg_position"] <= 10
        and row["content_age_days"] >= 180
    ):
        reasons.append("page_one_decay_risk")

    if (
        row["impressions_90d"] >= 500
        and row["avg_position"] > 0
        and row["avg_position"] <= 20
        and row["ctr"] < 0.5
    ):
        reasons.append("low_ctr_visible_page")

    if (
        row["sessions_90d"] >= 30
        and (
            row["engagement_rate"] < 30
            or row["scroll_rate"] < 30
        )
    ):
        reasons.append("low_engagement_visible_page")

    return reasons


score_df["reason_codes"] = score_df.apply(
    get_reason_codes,
    axis=1
)

# -----------------------------
# 7. Action
# -----------------------------

score_df["action"] = np.where(
    score_df["reason_codes"].str.len() > 0,
    "review",
    "monitor"
)

# -----------------------------
# 8. Rank
# -----------------------------

score_df = score_df.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

score_df["rank"] = np.arange(1, len(score_df) + 1)

# -----------------------------
# 9. Export
# -----------------------------

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

output_columns = [
    "rank",
    "content_id",
    "baseline_action_score",
    "action",
    "reason_codes",
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "word_count",
    "days_since_last_update",
    "trend_direction",
]

queue = score_df[output_columns].copy()

queue["reason_codes"] = queue["reason_codes"].apply(
    lambda x: "|".join(x) if x else "none"
)

queue.to_csv(output_path, index=False)

print("Rows ranked:", len(queue))
print("Output:", output_path)
print("\nTop 10:")
display(queue.head(10))


Rows ranked: 30000
Output: work/outputs/baseline_action_score.csv

Top 10:


,rank,content_id,baseline_action_score,action,reason_codes,impressions_90d,sessions_90d,ctr,avg_position,word_count,days_since_last_update,trend_direction
0,1,content_9532f197bbc8,75.402216,review,declining_with_demand|page_one_decay_risk|low_...,309192,1098,0.87,2.0,NaN,104,down
1,2,content_5fe46e04994d,74.306452,review,declining_with_demand|page_one_decay_risk|low_...,517715,520,0.14,4.2,NaN,104,down
2,3,content_2c2606c5d176,73.026128,review,declining_with_demand|page_one_decay_risk|low_...,347399,2146,0.53,4.2,NaN,104,down
3,4,content_3430a8b94511,71.511409,review,page_one_decay_risk|low_ctr_visible_page|low_e...,152617,534,0.29,3.3,NaN,104,stable
4,5,content_cbd93118300b,71.353560,review,declining_with_demand|page_one_decay_risk|low_...,145292,535,0.46,3.3,NaN,104,down
5,6,content_01908772c6db,71.303737,review,declining_with_demand|page_one_decay_risk|low_...,187893,782,0.45,4.0,NaN,104,down
6,7,content_4d1fe5b32dc2,71.089813,review,page_one_decay_risk|low_engagement_visible_page,97999,549,0.52,2.5,NaN,104,stable
7,8,content_e5ae436f9a16,71.053798,review,page_one_decay_risk|low_ctr_visible_page|low_e...,117741,522,0.45,3.0,NaN,104,stable
8,9,content_07f2e7a6f38a,70.939090,review,page_one_decay_risk|low_engagement_visible_page,101078,780,0.85,2.7,NaN,104,stable
9,10,content_79b25654070a,70.928766,review,page_one_decay_risk|low_ctr_visible_page|low_e...,148737,619,0.48,3.7,NaN,104,stable


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 review

I reviewed the top 20 rows as a decision-support queue rather than treating the ranking as ground truth.

The confidence note reflects the amount and quality of observed evidence. Higher impressions or sessions give more evidence than very low-volume pages.

A weak pick can still appear near the top because the baseline is deliberately simple. A page may have high visibility and freshness risk but still not be a good refresh candidate if the underlying measurements are noisy, the page has unusual seasonality, or the apparent issue is caused by another factor.

For each page, I record what would make the recommendation wrong. This keeps the score directional rather than presenting it as a guaranteed action.

In [5]:
# ML-07 — Section 3
# Generate a review table for the top 20.

top20 = score_df.head(20).copy()

def confidence_note(row):
    if row["impressions_90d"] >= 500 and row["sessions_90d"] >= 30:
        return "Higher evidence: meaningful observed volume"
    elif row["impressions_90d"] >= 250:
        return "Moderate evidence: observed impressions are sufficient for directional review"
    else:
        return "Lower evidence: review volume before acting"


def what_would_make_it_wrong(row):
    reasons = []

    if row["impressions_90d"] < 250:
        reasons.append("low impression volume")

    if row["sessions_90d"] < 30:
        reasons.append("low session volume")

    if row["avg_position"] <= 0:
        reasons.append("missing/invalid position")

    if not reasons:
        reasons.append(
            "the observed pattern may not persist or may have another explanation"
        )

    return "; ".join(reasons)


review20 = top20[
    [
        "rank",
        "content_id",
        "baseline_action_score",
        "action",
        "reason_codes",
        "impressions_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
    ]
].copy()

review20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
).values

review20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
).values

display(review20)


,rank,content_id,baseline_action_score,action,reason_codes,impressions_90d,sessions_90d,ctr,avg_position,confidence_note,what_would_make_it_wrong
0,1,content_9532f197bbc8,75.402216,review,"[declining_with_demand, page_one_decay_risk, l...",309192,1098,0.87,2.0,Higher evidence: meaningful observed volume,the observed pattern may not persist or may ha...
1,2,content_5fe46e04994d,74.306452,review,"[declining_with_demand, page_one_decay_risk, l...",517715,520,0.14,4.2,Higher evidence: meaningful observed volume,the observed pattern may not persist or may ha...
2,3,content_2c2606c5d176,73.026128,review,"[declining_with_demand, page_one_decay_risk, l...",347399,2146,0.53,4.2,Higher evidence: meaningful observed volume,the observed pattern may not persist or may ha...
3,4,content_3430a8b94511,71.511409,review,"[page_one_decay_risk, low_ctr_visible_page, lo...",152617,534,0.29,3.3,Higher evidence: meaningful observed volume,the observed pattern may not persist or may ha...
4,5,content_cbd93118300b,71.353560,review,"[declining_with_demand, page_one_decay_risk, l...",145292,535,0.46,3.3,Higher evidence: meaningful observed volume,the observed pattern may not persist or may ha...
5,6,content_01908772c6db,71.303737,review,"[declining_with_demand, page_one_decay_risk, l...",187893,782,0.45,4.0,Higher evidence: meaningful observed volume,the observed pattern may not persist or may ha...
6,7,content_4d1fe5b32dc2,71.089813,review,"[page_one_decay_risk, low_engagement_visible_p...",97999,549,0.52,2.5,Higher evidence: meaningful observed volume,the observed pattern may not persist or may ha...
7,8,content_e5ae436f9a16,71.053798,review,"[page_one_decay_risk, low_ctr_visible_page, lo...",117741,522,0.45,3.0,Higher evidence: meaningful observed volume,the observed pattern may not persist or may ha...
8,9,content_07f2e7a6f38a,70.939090,review,"[page_one_decay_risk, low_engagement_visible_p...",101078,780,0.85,2.7,Higher evidence: meaningful observed volume,the observed pattern may not persist or may ha...
9,10,content_79b25654070a,70.928766,review,"[page_one_decay_risk, low_ctr_visible_page, lo...",148737,619,0.48,3.7,Higher evidence: meaningful observed volume,the observed pattern may not persist or may ha...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks + leakage check

I do not assume that a high baseline score means the page definitely needs a refresh.

I flag weak picks where the score is driven by limited evidence, especially low impressions or sessions.

I also check that no product decision fields entered the feature/score calculation and that the baseline does not use future-window measurements.

This starter baseline uses the current 90-day snapshot. Therefore it should be described as a current-window prioritization rule, not as a future prediction model.

In [6]:
# ML-07 — Section 4
# Identify potentially weak top-20 picks.

weak_picks = review20[
    (review20["impressions_90d"] < 250)
    | (review20["sessions_90d"] < 30)
].copy()

print("Potentially weak top-20 picks:", len(weak_picks))

if len(weak_picks) > 0:
    display(weak_picks)
else:
    print("No top-20 picks were flagged by the simple low-volume check.")


# -----------------------------
# Leakage check
# -----------------------------

forbidden_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "is_quick_win",
]

used_score_columns = [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "word_count",
]

leaked_into_score = [
    col for col in forbidden_columns
    if col in used_score_columns
]

print("\nForbidden fields used in score:", leaked_into_score)

assert not leaked_into_score, (
    "Leakage detected in baseline score!"
)

print("\nLeakage check passed.")
print("No product decision fields or target-derived fields are used in the score.")
print(
    "Important limitation: this baseline uses the current-window snapshot, "
    "so it is not a strict future-window prediction."
)


Potentially weak top-20 picks: 0
No top-20 picks were flagged by the simple low-volume check.

Forbidden fields used in score: []

Leakage check passed.
No product decision fields or target-derived fields are used in the score.
Important limitation: this baseline uses the current-window snapshot, so it is not a strict future-window prediction.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.